[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burzynski-lab/OU_PHYS2222_ComputationalPhysics/blob/main/LectureNotes/16_ML_scikit_learn.ipynb)

# Machine Learning I: Regression and Classification

**PHYS 2222: Computational Physics** · University of Oklahoma · Spring 2027 · Week 13

> **Before you start:** in Colab, choose **File → Save a copy in Drive** so your changes are kept.

In Unit 3 you fit models to data: you wrote down a function with a few parameters (a straight line, an exponential, a Breit–Wigner peak), defined a loss ($\chi^2$ or $-2\ln L$), and let the computer find the parameters that minimize it. **Machine learning (ML) is the same idea, scaled up.** The model has many parameters (sometimes millions), it is usually not derived from physics, and the questions you care about shift from "what is the best-fit value of this parameter?" to "how well does this model predict data it has never seen?"

This week you will use **scikit-learn** (`sklearn`), the standard Python library for "classical" machine learning. Next week you will build neural networks with PyTorch.

## Learning goals

After this notebook you should be able to:

- Describe supervised vs unsupervised learning, and regression vs classification, using the words *features*, *labels*, and *loss*.
- Explain why ordinary least-squares fitting is already a machine-learning method, and do it with `LinearRegression`.
- Use the scikit-learn `fit` / `predict` pattern and build a `Pipeline`.
- Split data into training, validation, and test sets, and recognize overfitting and underfitting.
- Use cross-validation to choose a model, and regularization (`Ridge`) to tame an over-flexible one.
- Explain why and how to scale features with `StandardScaler`.
- Train logistic regression, decision tree, and gradient-boosted tree classifiers on real gamma-ray telescope data.
- Evaluate a classifier with a confusion matrix, an ROC curve, and AUC, and translate these into the physicist's language of **signal efficiency** and **background rejection**.
- Compute permutation feature importances, and explain why they must be interpreted carefully.

## 1. What is machine learning?

A useful working definition for this course:

> **A machine-learning model is a function with many adjustable parameters. "Learning" means choosing those parameters by minimizing a loss function on example data.**

That sentence also describes `curve_fit`. The differences are mostly of degree and of emphasis:

| | Fitting (Unit 3) | Machine learning |
|---|---|---|
| Model | Derived from physics, few parameters | Generic and flexible, many parameters |
| Goal | Measure the parameters (with uncertainties) | Make good predictions on new data |
| Key worry | Is the model correct? Is $\chi^2$/ndf reasonable? | Does it **generalize** beyond the training data? |

### Vocabulary

- **Features** (inputs, $\mathbf{x}$): the quantities you measure, for example the length and width of a shower image. A data set with $N$ examples and $d$ features is an array of shape `(N, d)`, usually called `X`.
- **Labels** (targets, $y$): the answer you want the model to produce, for example "gamma ray" or "hadron", or the period of a pendulum. Usually a 1D array `y` of length $N$.
- **Loss**: a number that measures how wrong the model is on the data. Mean squared error is the ML name for (unweighted) $\chi^2/N$.

**Notation.** A bold letter like $\mathbf{x}$ means a **vector**: just a list of numbers, $\mathbf{x} = (x_1, x_2, \dots, x_d)$, here the $d$ features of one example. In code it is a 1D NumPy array. The **dot product** of two vectors of the same length is the sum of their element-by-element products, $\mathbf{w}\cdot\mathbf{x} = w_1x_1 + w_2x_2 + \dots + w_dx_d$ (in NumPy: `np.dot(w, x)` or `w @ x`). A table of numbers with rows and columns, like `X`, is called a **matrix**. That's all the linear algebra you need this week.

### Kinds of learning

- **Supervised learning:** you have labeled examples $(\mathbf{x}_i, y_i)$ and want to learn the map $\mathbf{x}\mapsto y$.
  - **Regression:** $y$ is a continuous number (energy, period, mass).
  - **Classification:** $y$ is a category (gamma or hadron; electron, muon, or pion).
- **Unsupervised learning:** you only have $\mathbf{x}$, no labels. The goal is to find structure: clusters, low-dimensional patterns, or unusual events. You will see examples in notebook 18.

Everything in this notebook is supervised learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import constants

rng = np.random.default_rng(2222)

## 2. Linear regression is machine learning

Start with something you already know how to do. Quick physics recap: in the **photoelectric effect**, light of frequency $f$ knocks electrons out of a metal. Each photon carries energy $hf$; the electron needs at least the **work function** $\phi$ to escape, so the fastest electrons leave with kinetic energy $hf - \phi$. The **stopping voltage** $V_s$ is the reverse voltage needed to stop them, $eV_s = hf - \phi$, so it depends linearly on the frequency:

$$V_s = \frac{h}{e}\,f - \frac{\phi}{e},$$

Below we **simulate** 40 measurements for a metal with $\phi = 2.3$ eV, with 0.05 V of Gaussian noise on each voltage. (The real Millikan data you used in notebook 05 has only 6 points; simulating lets us control everything.)

In [ ]:
h_over_e = constants.h / constants.e      # V·s
phi_eV = 2.3                              # work function in eV (so phi/e in volts is 2.3 V)

f = rng.uniform(6e14, 12e14, size=40)     # frequencies in Hz
V = h_over_e * f - phi_eV + rng.normal(0, 0.05, size=f.size)   # stopping voltage in V

fig, ax = plt.subplots()
ax.plot(f / 1e14, V, "o", label="simulated measurements")
ax.set_xlabel(r"frequency $f$ ($10^{14}$ Hz)")
ax.set_ylabel(r"stopping voltage $V_s$ (V)")
ax.legend()
plt.show()

First, the Unit 3 way: `np.polyfit` minimizes the sum of squared residuals.

In [ ]:
slope, intercept = np.polyfit(f, V, 1)
print(f"polyfit: slope = {slope:.4e} V·s, intercept = {intercept:.3f} V")
print(f"true h/e = {h_over_e:.4e} V·s")

Now the scikit-learn way. Every scikit-learn model follows the same three-step pattern:

1. **Create** the model object: `model = LinearRegression()`. This only sets options; no data is involved yet.
2. **Fit** it to training data: `model.fit(X, y)`. This finds the parameters by minimizing the loss.
3. **Predict** on new inputs: `model.predict(X_new)`.

One detail trips everyone up the first time: scikit-learn always wants `X` to be **2D**, with shape `(n_samples, n_features)`, even when there is only one feature. Our `f` has shape `(40,)`. The method `.reshape(-1, 1)` turns it into shape `(40, 1)`: "one column, and as many rows as needed" (the `-1` means "work it out").

In [ ]:
from sklearn.linear_model import LinearRegression

X = f.reshape(-1, 1)          # shape (40, 1): 40 samples, 1 feature
print("f.shape =", f.shape, "  X.shape =", X.shape)

model = LinearRegression()    # 1. create
model.fit(X, V)               # 2. fit
print("coef_      =", model.coef_)        # one slope per feature
print("intercept_ =", model.intercept_)

The fitted parameters live in attributes whose names end with an underscore (`coef_`, `intercept_`). That is a scikit-learn convention meaning "this was learned from data". The numbers are identical to `polyfit`, because **`LinearRegression` is exactly least squares**: it minimizes the mean squared error

$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}\big(y_i - \hat y_i\big)^2, \qquad \hat y_i = w\,x_i + b,$$

which is $\chi^2/N$ with all uncertainties equal. Let's use the model to predict and to extract Planck's constant.

In [ ]:
f_new = np.array([[7e14], [1.0e15]])      # 2D again: 2 samples, 1 feature
print("predicted V_s:", model.predict(f_new), "V")

h_fit = model.coef_[0] * constants.e
print(f"h from fit = {h_fit:.3e} J·s   (CODATA: {constants.h:.3e} J·s)")
print(f"work function = {-model.intercept_:.3f} eV")

Notice what scikit-learn does *not* give you: parameter uncertainties, $\chi^2$/ndf, or a covariance matrix. That is the "prediction, not measurement" emphasis of ML. When you want to *measure* a parameter, `curve_fit` from Unit 3 is still the right tool.

## 3. How good is a model? Training and test sets

How well does a model predict data it has **not** seen? You can't answer that using the same data you fit to, because a flexible model can memorize its training data. The standard remedy is to hold some data back:

- **Training set**: used to fit the parameters.
- **Test set**: used *once*, at the end, to estimate performance on new data.

`train_test_split` shuffles the data randomly and splits it. `test_size=0.25` puts 25% in the test set; `random_state` is a seed so the split is reproducible.

The simulated data set below is a damped oscillation $x(t) = e^{-t/2}\cos(2\pi t)$ (in arbitrary units) "measured" at 40 random times with noise of 0.15. Pretend you don't know the true function and must learn it from the data.

In [ ]:
from sklearn.model_selection import train_test_split

def true_signal(t):
    return np.exp(-t / 2) * np.cos(2 * np.pi * t)

t = np.sort(rng.uniform(0, 2, size=40))                  # time in s
x = true_signal(t) + rng.normal(0, 0.15, size=t.size)    # position, arbitrary units

t_train, t_test, x_train, x_test = train_test_split(t, x, test_size=0.25, random_state=2)
print(len(t_train), "training points,", len(t_test), "test points")

In [ ]:
t_fine = np.linspace(0, 2, 400)

fig, ax = plt.subplots()
ax.plot(t_fine, true_signal(t_fine), "k--", label="true function")
ax.plot(t_train, x_train, "o", label="training data")
ax.plot(t_test, x_test, "s", label="test data")
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel("position $x$ (arb. units)")
ax.legend()
plt.show()

## 4. Overfitting and underfitting

We'll model $x(t)$ with a polynomial of degree $p$. In scikit-learn this is done in two steps:

1. `PolynomialFeatures(degree=p)` turns each input $t$ into the features $(1, t, t^2, \dots, t^p)$.
2. `LinearRegression()` fits a linear combination of those features. (A polynomial is *linear in its coefficients*, so this is still least squares.)

`make_pipeline` chains steps into a single **pipeline** object that has its own `fit` and `predict`. Calling `fit` runs the data through each step in order.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

def poly_model(degree):
    return make_pipeline(PolynomialFeatures(degree), LinearRegression())

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_train, x_train, "ko", label="training data")
for degree in [1, 6, 15]:
    m = poly_model(degree).fit(t_train.reshape(-1, 1), x_train)
    ax.plot(t_fine, m.predict(t_fine.reshape(-1, 1)), label=f"degree {degree}")
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel("position $x$ (arb. units)")
ax.legend()
plt.show()

Three different failure and success modes:

- **Degree 1 underfits.** A straight line is too simple to follow the oscillation. It is wrong on the training data *and* on new data. This is called high **bias**.
- **Degree 6** captures the shape reasonably well.
- **Degree 15 overfits.** It wiggles to pass close to every training point, including the noise, and goes wild between points and near the edges. It is great on the training data and bad on new data. This is called high **variance**.

To see this quantitatively, compute the mean squared error on the training and test sets for each degree. `mean_squared_error(y_true, y_pred)` does exactly what its name says.

In [ ]:
from sklearn.metrics import mean_squared_error

degrees = np.arange(0, 16)
mse_train, mse_test = [], []
for degree in degrees:
    m = poly_model(degree).fit(t_train.reshape(-1, 1), x_train)
    mse_train.append(mean_squared_error(x_train, m.predict(t_train.reshape(-1, 1))))
    mse_test.append(mean_squared_error(x_test, m.predict(t_test.reshape(-1, 1))))

fig, ax = plt.subplots()
ax.semilogy(degrees, mse_train, "o-", label="training MSE")
ax.semilogy(degrees, mse_test, "s-", label="test MSE")
ax.axhline(0.15**2, color="gray", ls=":", label=r"noise variance $\sigma^2$")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("mean squared error")
ax.legend()
plt.show()

This is the most important plot of the week:

- The **training error always goes down** as you add parameters. More freedom can only fit the training data better. (Same as $\chi^2$ always decreasing when you add fit parameters.)
- The **test error goes down, then up.** The minimum is the sweet spot between underfitting and overfitting.
- A training error far *below* the noise level $\sigma^2$ is a red flag: the model is fitting noise. In Unit 3 language, $\chi^2/\text{ndf} \ll 1$.

With only 10 test points, the test curve is itself noisy. That is a real problem in practice, and the next section addresses it.

### Training, validation, and test sets

There is a subtle trap here. If you **choose the degree** by looking at the test error, you have used the test set to make a decision, and its error is no longer an honest estimate of performance on new data. The careful procedure uses three sets:

| Set | Used for |
|---|---|
| Training | Fitting the model parameters (the polynomial coefficients) |
| Validation | Choosing **hyperparameters**: settings you pick rather than fit, like the degree |
| Test | Final, one-time estimate of performance. Look at it last. |

A **hyperparameter** is any choice that isn't learned by `fit`: polynomial degree, tree depth, regularization strength, network size.

## 5. Cross-validation

With small data sets, a single validation split wastes data and gives a noisy answer. **$k$-fold cross-validation** fixes both problems:

1. Split the training data into $k$ equal parts ("folds").
2. For each fold: train on the other $k-1$ folds, evaluate on this one.
3. Average the $k$ validation scores.

Every point is used for validation exactly once, and the average is much less noisy. `cross_val_score(model, X, y, cv=5, scoring=...)` does the whole loop. One quirk: scikit-learn always *maximizes* scores, so the MSE scoring option is called `"neg_mean_squared_error"` and returns $-\text{MSE}$. We flip the sign back.

`KFold(n_splits=5, shuffle=True, random_state=0)` makes the folds random (important here because our `t` values are sorted).

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=0)
cv_mse = []
for degree in degrees:
    scores = cross_val_score(poly_model(degree), t_train.reshape(-1, 1), x_train,
                             cv=cv, scoring="neg_mean_squared_error")
    cv_mse.append(-scores.mean())

best_degree = degrees[np.argmin(cv_mse)]
print("degree chosen by cross-validation:", best_degree)

fig, ax = plt.subplots()
ax.semilogy(degrees, cv_mse, "o-", label="5-fold CV MSE (training set only)")
ax.semilogy(degrees, mse_test, "s--", alpha=0.6, label="test MSE")
ax.set_xlabel("polynomial degree")
ax.set_ylabel("mean squared error")
ax.legend()
plt.show()

Cross-validation used only the training set, yet its curve has the same shape as the test curve. Now that the degree is chosen, fit once on the full training set and report the test error a single time:

In [ ]:
final = poly_model(best_degree).fit(t_train.reshape(-1, 1), x_train)
print(f"degree {best_degree}: test MSE = {mean_squared_error(x_test, final.predict(t_test.reshape(-1, 1))):.4f}"
      f"   (noise variance = {0.15**2:.4f})")

### Your turn 1

Repeat the overfitting study with **more data**: simulate 400 points instead of 40 (same noise), split them, and make the training/test MSE vs degree plot again. How does the gap between training and test error at degree 15 change? What does that tell you about the relationship between data set size and how complex a model you can afford?

In [ ]:
# Your code here

## 6. Regularization

Instead of limiting the number of parameters, you can keep a flexible model but **penalize large parameters**. Ridge regression minimizes

$$L(\mathbf{w}) = \sum_i \big(y_i - \hat y_i\big)^2 + \alpha \sum_j w_j^2 .$$

The penalty strength $\alpha$ is a hyperparameter. Large $\alpha$ forces small coefficients and a smoother curve; $\alpha \to 0$ gives back ordinary least squares.

For the penalty to treat all coefficients fairly, the features $t, t^2, \dots, t^{15}$ should be on similar scales, so we insert a `StandardScaler` (explained in Section 7) into the pipeline.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(t_train, x_train, "ko", label="training data")
for alpha in [1e-12, 1e-3, 1.0]:
    m = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=alpha))
    m.fit(t_train.reshape(-1, 1), x_train)
    mse = mean_squared_error(x_test, m.predict(t_test.reshape(-1, 1)))
    ax.plot(t_fine, m.predict(t_fine.reshape(-1, 1)), label=rf"$\alpha$ = {alpha:g}, test MSE = {mse:.3f}")
ax.set_ylim(-1.5, 1.5)
ax.set_xlabel("time $t$ (s)")
ax.set_ylabel("position $x$ (arb. units)")
ax.set_title("degree-15 polynomial with ridge regularization")
ax.legend(fontsize=9)
plt.show()

With a tiny $\alpha = 10^{-12}$ (essentially no regularization) the degree-15 polynomial still overfits. With a moderate $\alpha$ it behaves well, even though it has just as many parameters. With too large an $\alpha$ it is forced to be too smooth and underfits. Regularization is everywhere in ML; the neural networks next week use the same idea (it's called "weight decay" there).

### Your turn 2

Use cross-validation (on the training set only) to choose $\alpha$ for the degree-15 ridge pipeline. Try `alphas = np.logspace(-8, 2, 21)`, plot the CV mean squared error vs $\alpha$ on log-log axes, and report the best $\alpha$ and its test MSE.

In [ ]:
# Your code here

## 7. Feature scaling

Real features come in wildly different units and sizes: a shower length in mm (tens to hundreds), an angle in degrees (0–90), a ratio between 0 and 1. Many algorithms (regularized regression, logistic regression, neural networks, anything using distances) behave badly when features differ in scale by orders of magnitude.

`StandardScaler` transforms each feature to zero mean and unit standard deviation:

$$x' = \frac{x - \mu}{\sigma},$$

where $\mu$ and $\sigma$ are computed **from the training data only**. This rule matters: the scaler has "parameters" ($\mu$, $\sigma$) and fitting them on the test data would leak information about the test set into the model. Putting the scaler inside a pipeline handles this automatically: `pipeline.fit` fits the scaler on the training data, and `pipeline.predict` reuses those same $\mu$, $\sigma$.

Scalers (and `PolynomialFeatures`) are **transformers**: besides `fit` they have `transform`, and `fit_transform` does both at once.

In [ ]:
demo = np.array([[150.0, 0.30], [40.0, 0.55], [90.0, 0.20], [220.0, 0.45]])   # e.g. length (mm), ratio
scaler = StandardScaler().fit(demo)
print("means:", scaler.mean_)
print("stds: ", scaler.scale_)
print(scaler.transform(demo).round(2))

Tree-based models (Section 9) do not care about scaling, because they only ever ask "is this feature above or below some threshold?". Scaling is harmless for them, though.

## 8. Real data: the MAGIC gamma-ray telescope

### The physics

Some astrophysical objects (supernova remnants such as the Crab Nebula, the jets of active galaxies) emit **gamma rays**: photons with energies around $10^{11}$ eV = 100 GeV, about $10^{11}$ times more energetic than visible light. They can't reach the ground. When one hits the upper atmosphere it converts into an electron–positron pair near an atomic nucleus ($\gamma \to e^+e^-$, "pair production", which is $E = mc^2$ in action). The electron and positron are deflected by nuclei and radiate new photons ("bremsstrahlung"), which pair-produce again, and so on. The result is a cascade of millions of particles called an **air shower**, about 10 km up.

These particles are so relativistic that they travel faster than light does *in air* (light in air moves at $c/n$, with refractive index $n \approx 1.0003$; nothing beats $c$ itself). A charged particle moving faster than the local speed of light emits **Cherenkov light**, the optical analog of a sonic boom, in a narrow cone around its direction of motion. That faint blue flash, lasting a few nanoseconds, lights up a pool on the ground roughly 250 m across. An **Imaging Atmospheric Cherenkov Telescope (IACT)** such as MAGIC (on La Palma, Canary Islands) collects this faint, nanosecond-long flash with a large mirror and records it with a fast camera of photomultiplier pixels.

The problem: for every gamma ray, the telescope sees hundreds to thousands of **cosmic-ray hadrons** (mostly protons) that also make air showers. Separating gamma (signal) from hadron (background) showers is the central analysis task, and it's a classification problem.

What the images look like:

- **Gamma showers** are electromagnetic, compact, and regular. Their image is a narrow ellipse whose long axis points back toward the source position (the camera center, if the telescope points at the source).
- **Hadron showers** involve strong interactions, produce pions and sub-showers, and are broader and more irregular. Cosmic rays arrive from all directions, so their ellipses point anywhere.

In 1985 A. M. Hillas proposed summarizing each image by moments of its light distribution, now called **Hillas parameters**. The data set has ten of them:

| Column | Meaning |
|---|---|
| `fLength` | major axis of the ellipse (mm) |
| `fWidth` | minor axis of the ellipse (mm) |
| `fSize` | $\log_{10}$ of the total light in the image (photoelectrons); related to energy |
| `fConc` | fraction of light in the two brightest pixels |
| `fConc1` | fraction of light in the brightest pixel |
| `fAsym` | distance from the brightest pixel to the image center, along the major axis (mm) |
| `fM3Long` | cube root of the third moment along the major axis (mm) |
| `fM3Trans` | cube root of the third moment along the minor axis (mm) |
| `fAlpha` | angle between the major axis and the line to the camera center (degrees) |
| `fDist` | distance from the camera center to the image center (mm) |
| `class` | `g` = gamma (signal), `h` = hadron (background) |

An important caveat: these events are **Monte Carlo simulations** of the MAGIC telescope (made with the CORSIKA air-shower code), not recorded data. That is normal in physics: classifiers are trained on simulation, where the true label is known, and then applied to real data. Notebook 18 discusses what can go wrong with that.

Data source: Bock, R. (2004), *MAGIC Gamma Telescope*, UCI Machine Learning Repository, CC BY 4.0.

In [ ]:
import os
DATA_URL = "https://raw.githubusercontent.com/burzynski-lab/OU_PHYS2222_ComputationalPhysics/main/Data/"

def data_path(filename):
    """Return a local path to a course data file if it exists, otherwise its GitHub URL."""
    for folder in ["../Data", "Data", "."]:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return DATA_URL + filename

The file has no header row, so we pass `header=None` and supply column names with `names=`.

In [ ]:
columns = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym",
           "fM3Long", "fM3Trans", "fAlpha", "fDist", "class"]
magic = pd.read_csv(data_path("magic04.data"), header=None, names=columns)
print(magic.shape)
magic.head()

In [ ]:
print(magic["class"].value_counts())
magic.describe().round(2)

About 65% of the events are gammas. In real observations hadrons outnumber gammas by a huge factor, so this simulated sample is heavily enriched in signal. Keep that in mind when we talk about "accuracy".

### Exploring the features

Before training anything, look at the data. Histograms of each feature, separately for each class, show which features discriminate. `density=True` normalizes each histogram to unit area, so the shapes can be compared even though the classes have different sizes.

In [ ]:
gam = magic[magic["class"] == "g"]
had = magic[magic["class"] == "h"]

fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, name, unit in zip(axes.flat,
                          ["fAlpha", "fWidth", "fLength", "fSize", "fConc", "fDist"],
                          ["deg", "mm", "mm", r"log$_{10}$(p.e.)", "", "mm"]):
    lo, hi = np.percentile(magic[name], [0.5, 99.5])
    bins = np.linspace(lo, hi, 40)
    ax.hist(gam[name], bins=bins, density=True, histtype="step", lw=2, label="gamma")
    ax.hist(had[name], bins=bins, density=True, histtype="step", lw=2, label="hadron")
    ax.set_xlabel(f"{name} ({unit})" if unit else name)
axes[0, 0].legend()
fig.tight_layout()
plt.show()

The physics is visible directly:

- **`fAlpha`**: gamma images point at the source, so $\alpha$ piles up near 0°. Hadron images point randomly, so $\alpha$ is roughly flat. This is the classic gamma/hadron discriminant.
- **`fWidth`** and **`fLength`**: hadron images are broader and longer, as expected from their irregular development.
- **`fSize`** and **`fDist`** separate the classes less well on their own.

No single feature separates the classes cleanly. The job of the classifier is to combine all ten.

### Preparing features and labels

We encode the label as a number: $y = 1$ for gamma (signal), $y = 0$ for hadron (background). `stratify=y` in the split keeps the gamma fraction the same in the training and test sets.

In [ ]:
feature_names = columns[:-1]
X = magic[feature_names].to_numpy()
y = (magic["class"] == "g").astype(int).to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print("train:", X_train.shape, " test:", X_test.shape)
print("gamma fraction in train/test:", y_train.mean().round(3), y_test.mean().round(3))

## 9. Three classifiers

### Logistic regression

Logistic regression is the classification version of linear regression. It computes a linear combination of the features and squashes it through the **sigmoid** function into a probability:

$$P(\text{gamma}\mid\mathbf{x}) = \sigma(\mathbf{w}\cdot\mathbf{x} + b), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}.$$

The loss is the **binary cross-entropy**. Here is where it comes from. If the model says event $i$ is a gamma with probability $p_i$, then the probability of the observed label is $p_i$ if it really is a gamma ($y_i = 1$) and $1 - p_i$ if it is a hadron ($y_i = 0$). Treating events as independent, the probability of all the observed labels (the **likelihood**) is the product of these factors. Maximizing a product is awkward, so, as in notebook 15, we minimize minus its logarithm, which turns the product into a sum:

$$L = -\sum_i \Big[y_i \ln p_i + (1 - y_i)\ln(1 - p_i)\Big].$$

We put a `StandardScaler` in front, since the features have very different scales. `max_iter` raises the number of minimizer steps allowed.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_train, y_train)

print("training accuracy:", round(logreg.score(X_train, y_train), 4))
print("test accuracy:    ", round(logreg.score(X_test, y_test), 4))

For classifiers, `.score` returns the **accuracy**: the fraction of events classified correctly. 78% sounds decent, until you compare it with a "classifier" that ignores the data and always answers "gamma":

In [ ]:
print("always-gamma accuracy:", round(y_test.mean(), 4))

So accuracy alone can mislead, especially when classes are unbalanced. If the sample were 99.9% hadrons (closer to real observations), "always hadron" would be 99.9% accurate and completely useless. We need better metrics (Section 10).

The model can also output probabilities. `predict_proba` returns an array of shape `(N, 2)` with columns $P(\text{hadron})$ and $P(\text{gamma})$; `predict` simply thresholds $P(\text{gamma})$ at 0.5.

In [ ]:
print(logreg.predict_proba(X_test[:5]).round(3))
print(logreg.predict(X_test[:5]), " true:", y_test[:5])

### Decision tree

A decision tree asks a sequence of yes/no questions about single features ("is `fAlpha` < 20°?"), splitting the data into ever purer groups. Each leaf predicts the majority class of the training events that land in it. The hyperparameter `max_depth` limits how many questions in a row it can ask. Let's draw a depth-2 tree to see what it learned.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

small_tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(11, 5))
plot_tree(small_tree, feature_names=feature_names, class_names=["hadron", "gamma"],
          filled=True, impurity=False, ax=ax)
plt.show()

The tree discovered the physics on its own: its first question is about `fAlpha`, and the next ones about the shower length. Now compare trees of different depths on training and test accuracy:

In [ ]:
for depth in [2, 5, 10, 20, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f"max_depth={str(depth):>4}:  train acc = {tree.score(X_train, y_train):.3f}"
          f"   test acc = {tree.score(X_test, y_test):.3f}")

With no depth limit (`None`) the tree keeps splitting until every training event is classified correctly: 100% training accuracy, a perfect case of overfitting. A moderate depth does best on the test set.

### Gradient-boosted trees

A single tree is crude. **Gradient boosting** builds an ensemble: it fits a small tree, then fits a second tree to correct the first one's mistakes, then a third to correct what's left, and so on, adding up hundreds of small trees. (The "gradient" refers to how each new tree is fit to the gradient of the loss.) Boosted trees are among the most powerful methods for tabular data, and were the workhorse of particle-physics classification for years (the "BDT", boosted decision tree, in ROOT's TMVA).

scikit-learn's fast implementation is `HistGradientBoostingClassifier`. Its defaults are sensible.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

gbt = HistGradientBoostingClassifier(random_state=0)
gbt.fit(X_train, y_train)
print("training accuracy:", round(gbt.score(X_train, y_train), 4))
print("test accuracy:    ", round(gbt.score(X_test, y_test), 4))

## 10. Evaluating classifiers

### The confusion matrix

A confusion matrix counts the four possible outcomes. With gamma as the "positive" (signal) class:

| | predicted hadron | predicted gamma |
|---|---|---|
| **true hadron** | true negative (TN) | false positive (FP) |
| **true gamma** | false negative (FN) | true positive (TP) |

Physicists usually speak in rates instead:

- **Signal efficiency** = TP / (TP + FN): the fraction of real gammas you keep. (ML name: true positive rate, recall, sensitivity.)
- **Background efficiency** = FP / (FP + TN): the fraction of hadrons that sneak through. (ML name: false positive rate.)
- **Background rejection** = 1 − background efficiency.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = gbt.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print(f"signal efficiency     = {tp / (tp + fn):.3f}")
print(f"background efficiency = {fp / (fp + tn):.3f}")

In [ ]:
fig, ax = plt.subplots()
ConfusionMatrixDisplay(cm, display_labels=["hadron", "gamma"]).plot(ax=ax, colorbar=False)
ax.set_title("gradient-boosted trees, threshold 0.5")
plt.show()

### The ROC curve

Those numbers depend on the threshold 0.5, which is arbitrary. A gamma-ray astronomer studying a faint source might want very strong background rejection and accept losing signal; someone studying a bright flare might want high signal efficiency. Every threshold gives one (background efficiency, signal efficiency) pair. Scanning the threshold traces the **receiver operating characteristic (ROC) curve**. (The name comes from WWII radar operators.)

`roc_curve(y_true, scores)` does the scan and returns the false-positive rate, the true-positive rate, and the thresholds. The scores are the predicted probabilities of the positive class, `predict_proba(X)[:, 1]`.

The **area under the curve (AUC)** summarizes the whole curve in one number: 1 is perfect, 0.5 is random guessing. It equals the probability that a randomly chosen gamma gets a higher score than a randomly chosen hadron. It does **not** depend on the class balance, which makes it more meaningful than accuracy here.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

tree5 = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_train, y_train)
models = {"logistic regression": logreg, "decision tree (depth 5)": tree5, "gradient-boosted trees": gbt}

fig, ax = plt.subplots(figsize=(6, 5.5))
for name, m in models.items():
    score = m.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, score)
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {roc_auc_score(y_test, score):.3f})")
ax.plot([0, 1], [0, 1], "k:", label="random guessing")
ax.set_xlabel("background efficiency (false positive rate)")
ax.set_ylabel("signal efficiency (true positive rate)")
ax.legend(loc="lower right")
plt.show()

### Comparing with a traditional cut

Before ML, an IACT analysis might simply keep events with small $\alpha$. That cut is one point in the same plane. Let's see where the cut `fAlpha < 15°` lands, and plot in the form physicists often use: background *rejection* versus signal efficiency.

In [ ]:
alpha_test = X_test[:, feature_names.index("fAlpha")]
keep = alpha_test < 15
cut_sig_eff = keep[y_test == 1].mean()
cut_bkg_eff = keep[y_test == 0].mean()
print(f"cut fAlpha < 15 deg: signal eff = {cut_sig_eff:.3f}, background eff = {cut_bkg_eff:.3f}")

fig, ax = plt.subplots(figsize=(6, 5.5))
for name, m in models.items():
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test)[:, 1])
    ax.plot(tpr, 1 - fpr, label=name)
ax.plot(cut_sig_eff, 1 - cut_bkg_eff, "k*", ms=14, label=r"cut $\alpha < 15^\circ$")
ax.set_xlabel("signal efficiency")
ax.set_ylabel("background rejection")
ax.legend(loc="lower left")
plt.show()

At the same signal efficiency as the $\alpha$ cut, the boosted trees reject noticeably more background. That is the practical payoff of ML in physics: a better signal-to-background ratio from the same data, which means a more sensitive measurement.

To quote one number at a chosen working point, find the threshold where the signal efficiency crosses your target. `np.searchsorted` finds where a value would be inserted into a sorted array; the `tpr` from `roc_curve` is sorted increasingly.

In [ ]:
def bkg_eff_at(model, target_sig_eff):
    fpr, tpr, thr = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
    i = np.searchsorted(tpr, target_sig_eff)
    return fpr[i]

for name, m in models.items():
    print(f"{name:25s}: background efficiency at 50% signal efficiency = {bkg_eff_at(m, 0.5):.4f}")

### Your turn 3

Add a fourth classifier to the ROC comparison. Two good choices are `RandomForestClassifier` (from `sklearn.ensemble`; it averages many deep trees trained on random subsets of the data) and `KNeighborsClassifier` (from `sklearn.neighbors`; it classifies an event by majority vote of its nearest training events, so scaling matters: put it in a pipeline with `StandardScaler`). Report the AUC and the background efficiency at 70% signal efficiency for all four classifiers.

In [ ]:
# Your code here

## 11. Which features matter? Permutation importance

A natural question is which features the classifier relies on. **Permutation importance** answers it in a model-independent way: take the test set, randomly shuffle one feature column (destroying its relation to the label while keeping its distribution), and measure how much the score drops. A big drop means the model leaned on that feature. `n_repeats` repeats the shuffle to estimate the spread.

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(gbt, X_test, y_test, scoring="roc_auc",
                                n_repeats=5, random_state=0)
order = np.argsort(result.importances_mean)

fig, ax = plt.subplots()
ax.barh(np.array(feature_names)[order], result.importances_mean[order],
        xerr=result.importances_std[order])
ax.set_xlabel("drop in test AUC when feature is shuffled")
ax.set_title("permutation importance, gradient-boosted trees")
plt.show()

Reassuringly, `fAlpha` is at the top and the shower width and length are high up, matching the histograms and the physics. Interestingly, `fSize` also ranks high even though its histograms for the two classes looked similar: on its own it separates poorly, but *in combination* with the other features it helps (for example, how wide a gamma shower should be depends on its brightness, i.e. its energy). But be careful about over-interpreting such plots:

- **Correlated features share credit.** `fConc` and `fConc1` carry almost the same information. Shuffle one and the model can lean on the other, so *both* may look unimportant even though the information they carry matters.
- **It describes the model, not nature.** A low importance means *this trained model* doesn't use the feature, not that the feature is physically irrelevant. A different model may rank features differently.
- **Shuffling creates unphysical events.** A shuffled `fWidth` can be wider than the `fLength` of the same event, which never happens in real images; the model's behavior on such events is not very meaningful.
- **Importance is not causation.** Features can be predictive for reasons that have nothing to do with the physics you care about, e.g. a simulation artifact.

### Your turn 4

(a) Print the correlation matrix of the ten features with `magic[feature_names].corr().round(2)` and identify the most strongly correlated pairs. (b) Retrain the gradient-boosted classifier using only the **three** most important features from the plot above. How much AUC do you lose compared to all ten?

In [ ]:
# Your code here

### Your turn 5

Gradient boosting has hyperparameters too, such as `max_iter` (number of trees, default 100), `learning_rate` (default 0.1), and `max_depth`. Use `cross_val_score(..., cv=3, scoring="roc_auc")` on the **training set** to compare a few settings (e.g. `max_iter` in 50, 100, 300 and `learning_rate` in 0.05, 0.1, 0.3). Pick the best one, and only then evaluate it once on the test set. Is the improvement over the defaults bigger than the spread of the cross-validation scores?

In [ ]:
# Your code here

## Summary

- **Machine learning** = a flexible model with many parameters, fit by minimizing a loss on example data. Least squares is the simplest case.
- **Supervised** learning uses labeled examples (**regression** for numbers, **classification** for categories); **unsupervised** learning finds structure without labels.
- scikit-learn pattern: `model = Something(options)`, `model.fit(X_train, y_train)`, `model.predict(X_new)`, `model.predict_proba(X_new)`, `model.score(X, y)`. `X` is 2D with shape `(n_samples, n_features)`; use `.reshape(-1, 1)` for a single feature.
- `make_pipeline(PolynomialFeatures(p), StandardScaler(), Ridge(alpha))` chains transformers and a model into one object.
- `train_test_split(..., test_size, random_state, stratify)`: never judge a model on the data it was trained on. Training / validation / test sets have distinct roles; look at the test set last.
- **Underfitting** (too simple, high bias) vs **overfitting** (too flexible, high variance): training error always falls with complexity; test error has a minimum.
- **Cross-validation** (`KFold`, `cross_val_score`) chooses hyperparameters using only the training data.
- **Regularization** (`Ridge`) penalizes large parameters, taming over-flexible models.
- **Scale features** with `StandardScaler`, fit on training data only (pipelines do this for you).
- Classifiers: `LogisticRegression` (linear + sigmoid, cross-entropy loss = negative log-likelihood), `DecisionTreeClassifier`, `HistGradientBoostingClassifier` (boosted trees, excellent on tabular data).
- Accuracy can mislead. Use the **confusion matrix**, the **ROC curve** (`roc_curve`), and **AUC** (`roc_auc_score`); physicists speak of **signal efficiency** vs **background rejection** at a chosen working point.
- `permutation_importance` shows which features a model uses, but correlated features, unphysical shuffled events, and model dependence make it easy to over-interpret.

## Further reading

- scikit-learn User Guide and tutorials: https://scikit-learn.org/stable/user_guide.html
- J. VanderPlas, *Python Data Science Handbook*, Chapter 5 (Machine Learning), free online: https://jakevdp.github.io/PythonDataScienceHandbook/
- P. Mehta et al., "A high-bias, low-variance introduction to Machine Learning for physicists", *Physics Reports* 810 (2019), free at https://arxiv.org/abs/1803.08823
- MAGIC Gamma Telescope data set, UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope